# Lección 7 · Secuenciación: máquinas y barcos

**Qué vas a aprender**
- **Secuenciar**: decidir en qué **orden** una máquina (o un barco) hace sus trabajos.
- **Disyuntiva**: "i va antes que j, **o** j va antes que i". Cómo se escribe con una binaria y big-M.
- Por qué ese big-M es **débil**: el LP cree que la máquina hace todo a la vez.
- **CP-SAT**: un solver que razona con **intervalos** (`NoOverlap`), no con un LP.
- El taller crece de 5 a 40 trabajos: MILP contra CP-SAT con el mismo límite de tiempo.
- Barcos de instalación en el Báltico con un **modelo indexado en el tiempo**: una binaria por (proyecto, barco, día).
- Cuándo elegir MILP y cuándo CP.
- Escribir cada modelo en **notación matemática** y leerla símbolo por símbolo.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Hoy hay un solver nuevo: **CP-SAT**, de Google OR-Tools.

**Por qué las versiones fijas.** OR-Tools trae su propia copia de HiGHS por dentro. Con `highspy` 1.15 las dos copias chocan
y Python falla con `undefined symbol`. Con estas versiones conviven. Si Colab ya tenía otras, ve a **Entorno de ejecución → Reiniciar sesión** y corre todo de nuevo.

In [ ]:
%pip install -q pyomo "highspy==1.14.0" "ortools==9.15.6755"

In [ ]:
import io
import itertools
import logging
import math
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from ortools.sat.python import cp_model
from pyomo.repn import generate_standard_repn

SOLVER = pyo.SolverFactory("appsi_highs")
logging.getLogger("pyomo.contrib.appsi").setLevel(logging.ERROR)  # calla el aviso de "solución factible, no óptima" al cortar por tiempo


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"


def tamano(m):
    # El tamaño del modelo: variables, restricciones y no ceros (coeficientes distintos de 0)
    variables = list(m.component_data_objects(pyo.Var))
    restricciones = list(m.component_data_objects(pyo.Constraint, active=True))
    return {"variables": len(variables),
            "binarias": sum(v.is_binary() for v in variables),
            "restricciones": len(restricciones),
            "no_ceros": sum(sum(c != 0 for c in generate_standard_repn(r.body).linear_coefs) for r in restricciones)}

## 1. Dónde quedamos
- En la lección 3 viste que un big-M grande debilita la relajación LP: la cota se aleja y el branch & bound poda menos.
- En la lección 6 el tiempo era parte de la red: un nodo por bloque de 30 minutos.

Hoy las dos ideas se juntan en un problema nuevo: **el orden**. No decides cuánto ni quién, sino **quién va primero**.

**Analogía.** La mañana en casa, con un solo baño. Tú, tu esposa y tu hijo lo necesitan antes de salir.
El baño es una **máquina**: una persona a la vez. Entre tú y tu esposa solo hay dos opciones: entras tú primero **o** entra ella primero.
Si el orden sale mal, tu hijo llega tarde al *przedszkole*: eso es un **retraso**.

## 2. Una máquina, cuatro trabajos
El banco de **prueba de presión** del taller es uno solo. Hoy tiene 4 trabajos. Cada uno tiene su duración y la hora en que el cliente lo espera.

| Trabajo | Horas | Entrega (hora) |
|---|---|---|
| valvula_A | 4 | 6 |
| valvula_B | 2 | 3 |
| conector_C | 3 | 10 |
| brida_D | 5 | 8 |

El reloj empieza en 0. Si un trabajo termina después de su entrega, el **retraso** es la diferencia. Si termina antes, el retraso es 0 (no hay premio por llegar temprano).

**Las 3 piezas:**
- **Decido:** el orden de los 4 trabajos en el banco. Con el orden salen las horas de inicio.
- **Objetivo:** minimizar el retraso total.
- **Restricciones:** el banco hace un trabajo a la vez; un trabajo no se interrumpe.

### En notación
Conjunto: $J$ = trabajos. Datos: $p_j$ = horas (*processing time*), $d_j$ = entrega (*due date*).
Variables: $S_j$ = hora de inicio (*start*), $T_j$ = retraso (*tardiness*). El fin es $C_j = S_j + p_j$ (*completion*).

$$
\begin{aligned}
\min \quad & \sum_{j \in J} T_j \\
\text{s.a.} \quad & T_j \ge S_j + p_j - d_j && \forall\, j \in J && \text{(retraso)} \\
& \big( S_j \ge S_i + p_i \big) \;\lor\; \big( S_i \ge S_j + p_j \big) && \forall\, i, j \in J,\ i < j && \text{(disyuntiva)} \\
& S_j \ge 0, \quad T_j \ge 0 && \forall\, j \in J
\end{aligned}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $S_j$, $C_j$ | "ese de jota", "ce de jota" | hora de inicio y hora de fin | `S[j]`, `S[j] + p[j]` |
| $T_j \ge S_j + p_j - d_j$, $T_j \ge 0$ | "te de jota es al menos…" | $T_j = \max\{0,\ C_j - d_j\}$: el retraso, linealizado | `max(0, fin - entrega)` |
| $\lor$ | "o" | se cumple una de las dos (o las dos) | `or` |
| $i < j$ | "i menor que j" | cada par una sola vez: (A, B) sí, (B, A) no | `itertools.combinations(J, 2)` |

**Truco del retraso.** $T_j$ queda con dos pisos: $C_j - d_j$ y $0$. Como minimizamos, el solver lo baja hasta el piso más alto: el máximo.
Es la linealización de $\max$ de la lección 3, de este lado.

**La disyuntiva no es lineal.** Una restricción lineal es "esto **y** esto". Aquí dice "esto **o** esto". La sección 3 la convierte.

In [ ]:
TRABAJOS = pd.DataFrame({"trabajo": ["valvula_A", "valvula_B", "conector_C", "brida_D"],
                         "horas": [4, 2, 3, 5],
                         "entrega": [6, 3, 10, 8]})
p = TRABAJOS.set_index("trabajo")["horas"].to_dict()
d = TRABAJOS.set_index("trabajo")["entrega"].to_dict()


def retraso_total(orden):
    reloj, total = 0, 0
    for j in orden:
        reloj += p[j]
        total += max(0, reloj - d[j])
    return total


print("Orden de la tabla:", retraso_total(list(p)), "h de retraso")

### 2.1 Fuerza bruta
Con 4 trabajos hay $4! = 4 \cdot 3 \cdot 2 \cdot 1 = 24$ órdenes. Probamos todos.

In [ ]:
ordenes = sorted((retraso_total(o), o) for o in itertools.permutations(p))
print("Órdenes probados:", len(ordenes))
for r, o in ordenes[:3]:
    print(f"  {r:>2} h  {' → '.join(o)}")
print("  ...")
print(f"  {ordenes[-1][0]:>2} h  {' → '.join(ordenes[-1][1])}  (el peor)")

**Lectura:**
- El mejor orden es **valvula_B → valvula_A → conector_C → brida_D**, con **6 h** de retraso total. Es el único con 6 h.
- El peor tiene 17 h. El orden importa casi tres veces.
- La B va primero: es corta y vence a las 3. La D, larga, va al final aunque venza antes que la C.

**Los dos tamaños.** El espacio de búsqueda de un orden es $n!$ (*n factorial*). Crece **más rápido** que el $2^n$ de las binarias de la lección 3.

In [ ]:
for n in [4, 8, 10, 15, 20]:
    print(f"n = {n:>2}: {math.factorial(n):>26,} órdenes | 2^n = {2 ** n:>9,}".replace(",", "."))

Con 20 trabajos hay $20! \approx 2{,}4 \cdot 10^{18}$ órdenes. Probando mil millones por segundo, tardas 77 años.

## 3. La disyuntiva con big-M
La receta "either-or" de la lección 3: una binaria por par.

$y_{ij} = 1$ si $i$ va antes que $j$; $y_{ij} = 0$ si $j$ va antes que $i$.

$$
\begin{aligned}
& S_j \ge S_i + p_i - M\,(1 - y_{ij}) && \forall\, i < j && \text{(si } y_{ij} = 1\text{: } i \text{ antes)} \\
& S_i \ge S_j + p_j - M\, y_{ij} && \forall\, i < j && \text{(si } y_{ij} = 0\text{: } j \text{ antes)} \\
& y_{ij} \in \{0, 1\}
\end{aligned}
$$

- Con $y_{ij} = 1$: la primera queda $S_j \ge C_i$ (j empieza cuando i termina). La segunda queda $S_i \ge C_j - M$: no estorba si $M$ es grande.
- Con $y_{ij} = 0$: al revés.

**¿Qué M?** El más chico que sea válido. Ningún trabajo necesita empezar después de que el banco haya hecho todo: $M = \sum_j p_j = 14$ h, el **horizonte**.

In [ ]:
def una_maquina(trabajos, M=None, relajar=False):
    J = list(trabajos["trabajo"])
    p = dict(zip(J, trabajos["horas"]))
    d = dict(zip(J, trabajos["entrega"]))
    M = M or sum(p.values())  # horizonte
    m = pyo.ConcreteModel()
    m.J = pyo.Set(initialize=J)
    m.PARES = pyo.Set(initialize=list(itertools.combinations(J, 2)), dimen=2)  # i < j
    m.S = pyo.Var(m.J, within=pyo.NonNegativeReals)
    m.T = pyo.Var(m.J, within=pyo.NonNegativeReals)
    m.y = pyo.Var(m.PARES, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.obj = pyo.Objective(expr=sum(m.T[j] for j in m.J), sense=pyo.minimize)
    m.retraso = pyo.Constraint(m.J, rule=lambda m, j: m.T[j] >= m.S[j] + p[j] - d[j])
    m.antes = pyo.Constraint(m.PARES, rule=lambda m, i, j: m.S[j] >= m.S[i] + p[i] - M * (1 - m.y[i, j]))
    m.despues = pyo.Constraint(m.PARES, rule=lambda m, i, j: m.S[i] >= m.S[j] + p[j] - M * m.y[i, j])
    SOLVER.solve(m)
    return m


m1 = una_maquina(TRABAJOS)
print(f"Retraso total: {pyo.value(m1.obj):.0f} h")
print("Orden:", " → ".join(sorted(m1.J, key=lambda j: pyo.value(m1.S[j]))))
print("Tamaño:", tamano(m1))

**Lectura:**
- El MILP llega al mismo orden de la fuerza bruta: 6 h.
- 4 trabajos dan $\binom{4}{2} = 6$ pares: 6 binarias y 12 restricciones disyuntivas. Con $n$ trabajos son $n(n-1)/2$ binarias: crece **cuadrático**.

### 3.1 El big-M débil: el LP cree que la máquina hace todo a la vez
Relajamos las binarias a $[0, 1]$ y miramos la cota. Probamos con el M justo (14) y con M exagerados.

In [ ]:
for M in [14, 100, 10_000]:
    lp = una_maquina(TRABAJOS, M=M, relajar=True)
    inicios = {j: round(pyo.value(lp.S[j]), 1) + 0.0 for j in lp.J}
    print(f"M = {M:>6}: cota LP = {pyo.value(lp.obj):.1f} h | inicios {inicios}")

**Lectura:**
- La cota del LP es **0 h** con cualquier M. Hasta con el M más chico válido. El óptimo es 6 h: el gap en la raíz es del 100 %.
- Los inicios del LP son casi todos 0: **los 4 trabajos empiezan juntos**. El LP cree que el banco los hace a la vez.
- Por qué: con $y_{ij} = 0{,}5$, las dos restricciones se aflojan en $M/2 = 7$ h. Ningún trabajo dura más de 5 h: las dos se cumplen sin separar nada.

**Analogía.** En el LP, el baño tiene cortina: medio tú y media tu esposa adentro al mismo tiempo. Nadie espera, nadie llega tarde. La cota dice "0 retraso".

**La lección.** En la lección 3 el remedio era achicar M. Aquí no alcanza: la debilidad es de la **formulación**, no del número.
Con cota 0, el branch & bound no puede podar casi nada hasta bajar muy hondo en el árbol.

## 4. El taller completo: job shop
Ahora el taller tiene 3 máquinas: **torno**, **fresado** y **prueba**. Cada trabajo pasa por algunas, en un orden fijo: su **ruta**.

**Analogía.** La cocina de un restaurante. Cada plato tiene su receta en orden (cortar → cocinar → emplatar). Cada estación hace una cosa a la vez.

**Las 3 piezas:**
- **Decido:** el orden de las operaciones en cada máquina.
- **Objetivo:** terminar todo lo antes posible: minimizar el **makespan** (la hora en que termina el último trabajo).
- **Restricciones:** cada trabajo sigue su ruta (un paso empieza cuando termina el anterior); cada máquina hace una operación a la vez.

Los datos, en formato largo: una fila por **operación** (trabajo, paso).

In [ ]:
OPS = pd.read_csv(io.StringIO('''trabajo,paso,maquina,horas
valvula_1,0,torno,3
valvula_1,1,fresado,2
valvula_1,2,prueba,2
valvula_2,0,torno,2
valvula_2,1,prueba,1
valvula_2,2,fresado,4
conector_1,0,fresado,4
conector_1,1,prueba,3
conector_2,0,prueba,2
conector_2,1,torno,3
conector_2,2,fresado,1
forjado_1,0,torno,5
forjado_1,1,fresado,2
'''))
print("Operaciones:", len(OPS))
OPS.head()

### En notación
Conjuntos: $O$ = operaciones $(j, k)$: el paso $k$ del trabajo $j$. $K$ = máquinas.
Datos: $p_o$ = horas, $m_o$ = máquina de la operación $o$, $\text{últ}(j)$ = último paso de $j$.
Variables: $S_o$ = inicio, $C_{\max}$ = makespan, $y_{ab} \in \{0,1\}$ = 1 si $a$ va antes que $b$ en su máquina.

$$
\begin{aligned}
\min \quad & C_{\max} \\
\text{s.a.} \quad & S_{j,k} \ge S_{j,k-1} + p_{j,k-1} && \forall\, (j, k) \in O,\ k \ge 1 && \text{(ruta)} \\
& S_b \ge S_a + p_a - M\,(1 - y_{ab}) && \forall\, (a, b) \in D && \text{(a antes)} \\
& S_a \ge S_b + p_b - M\, y_{ab} && \forall\, (a, b) \in D && \text{(b antes)} \\
& C_{\max} \ge S_{j,\text{últ}(j)} + p_{j,\text{últ}(j)} && \forall\, j && \text{(fin)} \\
& S_o \ge 0, \quad y_{ab} \in \{0, 1\}
\end{aligned}
$$

con $D = \{\, (a, b) \in O \times O : a < b,\ m_a = m_b \,\}$: los pares de operaciones **que comparten máquina**. Es el truco de los pares factibles (lección 4):
dos operaciones en máquinas distintas nunca chocan, así que no llevan binaria.

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $C_{\max}$ | "ce máx" | makespan: el fin del último trabajo | `Cmax` |
| $D = \{(a, b) : m_a = m_b\}$ | "pares con la misma máquina" | solo los pares que pueden chocar | `[(a, b) for a, b in combinations(O, 2) if maq[a] == maq[b]]` |
| $(j, k)$ | "paso k del trabajo j" | una operación: un renglón de `OPS` | `(trabajo, paso)` |

In [ ]:
def jobshop_milp(ops, relajar=False, limite=None):
    O = list(zip(ops["trabajo"], ops["paso"]))
    p = dict(zip(O, ops["horas"]))
    maq = dict(zip(O, ops["maquina"]))
    ultimo = ops.groupby("trabajo")["paso"].max().to_dict()
    M = int(ops["horas"].sum())  # horizonte: todo en fila
    D = [(a, b) for a, b in itertools.combinations(O, 2) if maq[a] == maq[b]]

    m = pyo.ConcreteModel()
    m.O = pyo.Set(initialize=O, dimen=2)
    m.D = pyo.Set(initialize=[a + b for a, b in D], dimen=4)  # (j1, k1, j2, k2)
    m.S = pyo.Var(m.O, within=pyo.NonNegativeReals)
    m.Cmax = pyo.Var(within=pyo.NonNegativeReals)
    m.y = pyo.Var(m.D, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.obj = pyo.Objective(expr=m.Cmax, sense=pyo.minimize)
    m.ruta = pyo.Constraint([o for o in O if o[1] >= 1],
                            rule=lambda m, j, k: m.S[j, k] >= m.S[j, k - 1] + p[j, k - 1])
    m.antes = pyo.Constraint(m.D, rule=lambda m, j1, k1, j2, k2:
                             m.S[j2, k2] >= m.S[j1, k1] + p[j1, k1] - M * (1 - m.y[j1, k1, j2, k2]))
    m.despues = pyo.Constraint(m.D, rule=lambda m, j1, k1, j2, k2:
                               m.S[j1, k1] >= m.S[j2, k2] + p[j2, k2] - M * m.y[j1, k1, j2, k2])
    m.fin = pyo.Constraint(list(ultimo.items()), rule=lambda m, j, k: m.Cmax >= m.S[j, k] + p[j, k])

    inicio = time.perf_counter()
    res = SOLVER.solve(m, timelimit=limite)
    m.segundos = time.perf_counter() - inicio
    m.cota = res.problem.lower_bound  # best bound (minimizamos: la cota es por abajo)
    return m


mj = jobshop_milp(OPS)
lpj = jobshop_milp(OPS, relajar=True)
print(f"MILP: makespan {pyo.value(mj.Cmax):.0f} h en {mj.segundos:.2f} s")
print(f"LP:   cota {pyo.value(lpj.Cmax):.0f} h")
print("Tamaño:", tamano(mj))

Un **diagrama de Gantt** muestra el plan: una fila por máquina, una barra por operación.

In [ ]:
COLORES = plt.cm.tab20.colors


def gantt(plan, fila, titulo="", cerrado=None, ancho=10, eje="horas"):
    # plan: DataFrame con columnas [fila, "trabajo", "inicio", "fin"]
    filas = sorted(plan[fila].unique())
    color = {t: COLORES[i % 20] for i, t in enumerate(sorted(plan["trabajo"].unique()))}
    fig, ax = plt.subplots(figsize=(ancho, 0.6 * len(filas) + 1.2))
    for r in plan.itertuples():
        y = filas.index(getattr(r, fila))
        ax.barh(y, r.fin - r.inicio, left=r.inicio, color=color[r.trabajo], edgecolor="black")
        if r.fin - r.inicio >= 0.03 * plan["fin"].max():  # etiqueta solo en barras anchas
            ax.text((r.inicio + r.fin) / 2, y, r.trabajo, ha="center", va="center", fontsize=7)
    for dia in cerrado or []:
        ax.axvspan(dia, dia + 1, color="gray", alpha=0.25, lw=0)
    ax.set_yticks(range(len(filas)), filas)
    ax.invert_yaxis()
    ax.set(xlabel=eje, title=titulo)
    plt.tight_layout()
    plt.show()


plan_milp = OPS.assign(inicio=[pyo.value(mj.S[o]) for o in zip(OPS["trabajo"], OPS["paso"])])
plan_milp["fin"] = plan_milp["inicio"] + plan_milp["horas"]
gantt(plan_milp, "maquina", f"Job shop con MILP: makespan {pyo.value(mj.Cmax):.0f} h")

**Lectura:**
- El taller termina todo en **14 h**.
- La cota del LP es **7 h**: la ruta más larga (válvula 1, 3 + 2 + 2). El LP respeta las rutas, pero ignora que las máquinas se comparten. El big-M débil otra vez.

**Una cota de primaria.** El torno tiene 13 h de trabajo. Haga lo que haga, el taller no termina antes de 13 h.

In [ ]:
carga = OPS.groupby("maquina")["horas"].sum()
ruta = OPS.groupby("trabajo")["horas"].sum()
print("Horas por máquina:", carga.to_dict())
print("Horas por trabajo:", ruta.to_dict())
print("Cota simple = max(máquina más cargada, ruta más larga) =", max(carga.max(), ruta.max()), "h")

**Lectura:**
- La cota simple es **13 h**: torno y fresado tienen 13 h de trabajo cada uno. Está a 1 h del óptimo.
- El LP da 7 h. Una suma por máquina le gana al LP. Guarda esa idea: CP-SAT razona justo así.

**Los dos tamaños del job shop.**
- Modelo: una binaria por par que comparte máquina. Torno 4 operaciones, fresado 5, prueba 4: $6 + 10 + 6 = 22$ binarias.
- Espacio de búsqueda: un orden por máquina, $\prod_{k \in K} n_k!$. Con $n$ trabajos que pasan por las $m$ máquinas, $(n!)^m$.

In [ ]:
n_por_maquina = OPS["maquina"].value_counts()
print("Operaciones por máquina:", n_por_maquina.to_dict())
print("Binarias:", sum(math.comb(n, 2) for n in n_por_maquina))
print("Combinaciones de órdenes:", f"{math.prod(math.factorial(n) for n in n_por_maquina):,}".replace(",", "."))
print("10 trabajos × 5 máquinas: (10!)^5 ≈", f"{float(math.factorial(10)) ** 5:.1e}")

Muchas de esas combinaciones son imposibles (chocan con las rutas), pero el número muestra por qué no se prueba todo.
Con 10 trabajos y 5 máquinas, $(10!)^5 \approx 6 \cdot 10^{32}$.

## 5. CP-SAT: razonar con intervalos
**Programación por restricciones** (*constraint programming*, CP) es otra familia de solvers. No relaja nada a un LP. Trabaja así:
1. Cada variable tiene un **dominio**: los valores que todavía puede tomar. Al empezar, el inicio del forjado puede ser cualquier hora de 0 a 31.
2. **Propagación**: cada restricción tacha valores imposibles. "Si la válvula 1 usa el torno de 0 a 3, el forjado no empieza en el torno antes de 3".
3. Cuando ya no puede tachar más, **prueba** un valor y vuelve a propagar. Si llega a un callejón sin salida, **aprende** una regla nueva para no volver a entrar. Esa es la parte *SAT*.

**Analogía.** Un sudoku. No pruebas números al azar: tachas en cada casilla lo que ya no puede ir. Casi siempre, tachando, el sudoku se resuelve solo.

**El objeto clave: el intervalo.** Una variable de intervalo junta inicio, duración y fin. `NoOverlap` dice "estos intervalos no se pisan".
Es la disyuntiva **entera**, sin binarias ni M. Y el solver conoce su estructura: suma la carga de la máquina, como la cota simple de arriba.

### En notación (estilo CP)
$$
\begin{aligned}
\min \quad & C_{\max} \\
\text{s.a.} \quad & \text{ini}_{j,k} \ge \text{fin}_{j,k-1} && \forall\, (j, k) \in O,\ k \ge 1 && \text{(ruta)} \\
& \text{NoOverlap}\big(\{\, [\text{ini}_o, \text{fin}_o) : m_o = k \,\}\big) && \forall\, k \in K && \text{(máquina)} \\
& C_{\max} = \max_{o \in O} \text{fin}_o \\
& \text{fin}_o = \text{ini}_o + p_o, \quad \text{ini}_o \in \{0, 1, \dots, H\} && \forall\, o \in O
\end{aligned}
$$

| Símbolo | Se lee | Significa | En OR-Tools |
|---|---|---|---|
| $[a, b)$ | "intervalo de a a b, abierto en b" | ocupa desde $a$ hasta justo antes de $b$: uno puede empezar donde otro termina | `new_interval_var(ini, dur, fin)` |
| $\text{NoOverlap}(\cdot)$ | "sin solape" | ningún par de intervalos de la lista se pisa | `add_no_overlap([...])` |
| $\{0, 1, \dots, H\}$ | "enteros de 0 a H" | el dominio: CP-SAT solo trabaja con **enteros** | `new_int_var(0, H, ...)` |
| $\max_{o}$ | "el máximo sobre o" | CP-SAT lo acepta directo, sin linealizar | `add_max_equality(Cmax, [...])` |

**Ojo:** CP-SAT solo acepta enteros. Si tus horas tienen decimales, cambia de unidad (minutos) y redondea.

In [ ]:
def jobshop_cp(ops, limite=10, paradas=None):
    # paradas: {maquina: [(inicio, fin), ...]} tiempos en que la máquina no está disponible (lo usas en el ejercicio 1)
    O = list(zip(ops["trabajo"], ops["paso"]))
    p = dict(zip(O, ops["horas"]))
    maq = dict(zip(O, ops["maquina"]))
    H = int(ops["horas"].sum()) + sum(f - i for lista in (paradas or {}).values() for i, f in lista)

    m = cp_model.CpModel()
    ini = {o: m.new_int_var(0, H, f"ini_{o}") for o in O}
    fin = {o: m.new_int_var(0, H, f"fin_{o}") for o in O}
    intervalo = {o: m.new_interval_var(ini[o], int(p[o]), fin[o], f"int_{o}") for o in O}
    for j, k in O:
        if k >= 1:
            m.add(ini[j, k] >= fin[j, k - 1])  # ruta
    for k in ops["maquina"].unique():
        fijos = [m.new_fixed_size_interval_var(i, f - i, f"parada_{k}_{i}") for i, f in (paradas or {}).get(k, [])]
        m.add_no_overlap([intervalo[o] for o in O if maq[o] == k] + fijos)
    cmax = m.new_int_var(0, H, "Cmax")
    m.add_max_equality(cmax, [fin[o] for o in O])
    m.minimize(cmax)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = limite
    inicio = time.perf_counter()
    estado = solver.solve(m)
    plan = ops.assign(inicio=[solver.value(ini[o]) for o in O], fin=[solver.value(fin[o]) for o in O])
    return {"estado": solver.status_name(estado), "makespan": solver.objective_value,
            "cota": solver.best_objective_bound, "segundos": time.perf_counter() - inicio, "plan": plan}


cp = jobshop_cp(OPS)
print(f"CP-SAT: {cp['estado']}, makespan {cp['makespan']:.0f} h, cota {cp['cota']:.0f} h, {cp['segundos']:.3f} s")
gantt(cp["plan"], "maquina", f"Job shop con CP-SAT: makespan {cp['makespan']:.0f} h")

**Lectura:**
- CP-SAT llega al mismo makespan, **14 h**, y lo demuestra: `OPTIMAL` y cota 14. El orden en las máquinas puede ser otro: hay óptimos alternativos.
- El modelo es más corto: sin binarias, sin M, sin pares. Cada máquina es **una** línea: `add_no_overlap`.
- Con 13 operaciones los dos tardan milisegundos. La diferencia aparece cuando el taller crece.

## 6. El modelo crece: de 5 a 40 trabajos
Generamos talleres al azar (semilla fija) con 5 máquinas. Cada trabajo pasa por las 5 en un orden propio, de 1 a 9 h cada paso.
Los datos son una tabla larga, igual a `OPS`.

In [ ]:
def taller_aleatorio(n, n_maquinas=5, semilla=0):
    rng = np.random.default_rng(semilla)
    filas = [(f"T{j + 1:02d}", k, f"M{maq + 1}", int(rng.integers(1, 10)))
             for j in range(n) for k, maq in enumerate(rng.permutation(n_maquinas))]
    return pd.DataFrame(filas, columns=["trabajo", "paso", "maquina", "horas"])


taller_aleatorio(10).head(7)

Los dos solvers, con **10 segundos** cada uno. Para cada tamaño: la cota simple, lo mejor que encontró cada uno y su cota.
Esta celda tarda alrededor de 40 segundos.

In [ ]:
filas = []
for n in [5, 10, 20, 40]:
    ops_n = taller_aleatorio(n)
    simple = max(ops_n.groupby("maquina")["horas"].sum().max(), ops_n.groupby("trabajo")["horas"].sum().max())
    mm = jobshop_milp(ops_n, limite=10)
    cc = jobshop_cp(ops_n, limite=10)
    filas.append({"trabajos": n, "binarias_milp": tamano(mm)["binarias"], "cota_simple": simple,
                  "milp_mejor": round(pyo.value(mm.Cmax)), "milp_cota": round(mm.cota, 1),
                  "milp_gap_%": round(100 * (pyo.value(mm.Cmax) - mm.cota) / pyo.value(mm.Cmax), 1),
                  "milp_s": round(mm.segundos, 1),
                  "cp_mejor": round(cc["makespan"]), "cp_estado": cc["estado"], "cp_s": round(cc["segundos"], 2)})
escala = pd.DataFrame(filas)
escala

**Cómo se lee** (los segundos cambian con el computador; el patrón no):
- Con 5 trabajos, los dos llegan al óptimo en una fracción de segundo.
- Desde 10 trabajos, el MILP se corta a los 10 s. Su mejor plan es peor que el de CP-SAT (con 40 trabajos, unas 316 h contra 234 h).
- Su **cota se queda en 42 a 47 h** aunque el óptimo suba a 128 o 234 h. Con 40 trabajos el gap llega a 87 %.
  Esa cota es la del big-M débil: casi no sabe nada de las máquinas. La cota simple, una suma por máquina, es 5 veces mejor.
- CP-SAT demuestra el óptimo (`OPTIMAL`) en todos, en décimas de segundo.
- Con 20 y 40 trabajos el óptimo de CP-SAT es **igual a la cota simple**: la máquina más cargada manda, y CP-SAT lo ve de entrada.

En mi computador el MILP con 10 trabajos y **sin** límite de tiempo tardó unos 400 segundos en demostrar el óptimo (64 h). CP-SAT: 0,06 s.

### 6.1 La solución de 40 trabajos, de vuelta a tablas

In [ ]:
ops40 = taller_aleatorio(40)
cp40 = jobshop_cp(ops40)
plan40 = cp40["plan"]

uso = plan40.groupby("maquina").agg(horas=("horas", "sum"), primera=("inicio", "min"), ultima=("fin", "max"))
uso["uso_%"] = (100 * uso["horas"] / cp40["makespan"]).round(1)
uso["ocio_h"] = cp40["makespan"] - uso["horas"]
uso.sort_values("uso_%", ascending=False)

In [ ]:
# Top 5: los trabajos que terminan más tarde
fin_trabajo = plan40.groupby("trabajo").agg(termina=("fin", "max"), horas=("horas", "sum"))
fin_trabajo.nlargest(5, "termina")

In [ ]:
gantt(plan40, "maquina", f"40 trabajos × 5 máquinas: makespan {cp40['makespan']:.0f} h", ancho=13)

**Lectura para el jefe del taller:**
- Los 40 trabajos se terminan en **234 h**. No hay forma de bajar de ahí con estas máquinas: **M5** trabaja el 100 % del tiempo.
- M5 es el **cuello de botella**. Las demás tienen entre 13 y 39 h de ocio (columna `ocio_h`). T05 es el último en salir.
- Para terminar antes no sirve reordenar: hay que darle capacidad a la máquina al 100 % (otro turno, otra máquina, o mover operaciones a otra).
- Es la holgura de la lección 1, ahora en el tiempo.

## 7. Barcos de instalación en el Báltico: modelo indexado en el tiempo
Una empresa de servicios offshore instala tres parques eólicos en el Báltico (BW1, BW2, BW3). Tiene 3 barcos especiales:
uno de grúa pesada, un *jack-up* (se apoya en el fondo con patas) y uno que tiende cable. Cuestan decenas de miles de euros por día.

- Cada proyecto dura unos días seguidos y solo puede hacerlo ciertos barcos.
- Hay días de mal tiempo: nadie trabaja en el mar. Un proyecto no puede tener un día de mal tiempo en el medio.
- Si un proyecto termina después de su fecha, hay **multa** por día.
- Las turbinas de BW1 solo se montan cuando terminan sus cimentaciones.

**Analogía.** Reservar salas de reuniones en un calendario. Cada reunión (proyecto) necesita una sala compatible (barco) por días seguidos.
Una sala no tiene dos reuniones el mismo día. Algunos días el edificio cierra (clima).

**Las 3 piezas:**
- **Decido:** qué barco hace cada proyecto y qué día empieza.
- **Objetivo:** minimizar costo de barcos + multas por retraso.
- **Restricciones:** cada proyecto se hace una vez; un barco, un proyecto por día; sin días de mal tiempo en medio; turbinas después de cimentaciones.

Cuatro tablas, como en SQL. Costos en euros.

In [ ]:
proyectos = pd.read_csv(io.StringIO('''proyecto,dias,listo,entrega,multa_dia,costo_aplazar
cimentacion_BW1,6,0,12,40000,3000000
cimentacion_BW2,6,4,18,40000,3000000
turbinas_BW1,8,8,24,60000,3000000
subestacion_BW1,5,2,14,80000,3000000
cable_export_BW1,7,0,20,50000,3000000
cable_array_BW2,5,10,26,30000,3000000
reparacion_BW3,3,0,6,100000,3000000
cable_export_BW2,6,12,30,50000,3000000
'''))
barcos = pd.DataFrame({"barco": ["grua_A", "jackup_B", "cable_C"], "costo_dia": [90000, 60000, 35000]})
compat = pd.read_csv(io.StringIO('''proyecto,barco
cimentacion_BW1,grua_A
cimentacion_BW1,jackup_B
cimentacion_BW2,grua_A
cimentacion_BW2,jackup_B
turbinas_BW1,jackup_B
subestacion_BW1,grua_A
cable_export_BW1,cable_C
cable_array_BW2,cable_C
reparacion_BW3,grua_A
reparacion_BW3,jackup_B
reparacion_BW3,cable_C
cable_export_BW2,cable_C
'''))
DIAS = 36
MAL_TIEMPO = [7, 8, 15, 16, 17, 25, 26]
clima = pd.DataFrame({"dia": range(DIAS), "ventana": [t not in MAL_TIEMPO for t in range(DIAS)]})
PRECEDENCIAS = [("cimentacion_BW1", "turbinas_BW1")]  # (antes, después)

print(len(proyectos), "proyectos |", len(barcos), "barcos |", len(compat), "pares compatibles |", DIAS, "días,", len(MAL_TIEMPO), "de mal tiempo")
proyectos

### 7.1 La idea: una binaria por (proyecto, barco, día de inicio)
En vez de "¿quién va antes?", preguntamos "¿**cuándo** empieza y con qué barco?":

$x_{pvt} = 1$ si el proyecto $p$ empieza el día $t$ con el barco $v$.

Dos ventajas enormes:
1. **Todo lo que depende del día se calcula antes.** Clima, multa, costo: para cada $(p, v, t)$ es un número. El modelo no necesita big-M.
2. **Generar solo lo factible** (lecciones 4 y 6). Un inicio que pisa un día de mal tiempo, que empieza antes de que el proyecto esté listo
   o con un barco incompatible, **no se crea**.

Con un *join* y un filtro, como en SQL:

In [ ]:
def inicios_factibles(proyectos, barcos, compat, clima):
    T = len(clima)
    malo = set(clima.loc[~clima["ventana"], "dia"])
    k = (compat.merge(proyectos, on="proyecto").merge(barcos, on="barco")
         .merge(pd.DataFrame({"t": range(T)}), how="cross"))                     # todos los (p, v, t) compatibles
    k = k[(k["t"] >= k["listo"]) & (k["t"] + k["dias"] <= T)]                     # listo y dentro del horizonte
    k = k[[malo.isdisjoint(range(r.t, r.t + r.dias)) for r in k.itertuples()]]  # sin mal tiempo en medio
    k = k.assign(retraso=(k["t"] + k["dias"] - k["entrega"]).clip(lower=0))
    k = k.assign(costo=k["dias"] * k["costo_dia"] + k["retraso"] * k["multa_dia"])  # costo precalculado
    return k.reset_index(drop=True)


K = inicios_factibles(proyectos, barcos, compat, clima)
print(f"Inicios factibles: {len(K)} de {len(compat) * DIAS} (pares compatibles × días)")
K.sample(5, random_state=1)[["proyecto", "barco", "t", "dias", "retraso", "costo"]]

### En notación
Conjuntos: $P$ = proyectos, $V$ = barcos, $K \subseteq P \times V \times \{0, \dots, H-1\}$ = inicios factibles (la tabla `K`).
Datos: $c_{pvt}$ = costo del inicio (días × costo diario + días de retraso × multa), $\ell_p$ = días que dura, $a_p$ = costo de aplazarlo al mes siguiente.
Variables: $x_{pvt} \in \{0,1\}$ y $u_p \in \{0, 1\}$ = 1 si el proyecto se aplaza (el modelo elástico de la lección 4).

$$
\begin{aligned}
\min \quad & \sum_{(p,v,t) \in K} c_{pvt}\, x_{pvt} + \sum_{p \in P} a_p\, u_p \\
\text{s.a.} \quad & \sum_{(v,t) : (p,v,t) \in K} x_{pvt} + u_p = 1 && \forall\, p \in P && \text{(una vez)} \\
& \sum_{(p,s) : (p,v,s) \in K,\ s \le t < s + \ell_p} x_{pvs} \le 1 && \forall\, v \in V,\ t && \text{(un proyecto por barco y día)} \\
& \sum_{(v,t)} t\, x_{qvt} \ge \sum_{(v,t)} (t + \ell_p)\, x_{pvt} && \forall\, (p, q) \in \text{Prec} && \text{(q después de p)} \\
& x_{pvt} \in \{0, 1\}, \quad u_p \in \{0, 1\}
\end{aligned}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $K \subseteq P \times V \times \{0, \dots, H-1\}$ | "K, triples factibles" | solo los inicios posibles | la tabla `K` |
| $s \le t < s + \ell_p$ | "s hasta s más ele" | el proyecto que empezó el día $s$ sigue en el barco el día $t$ | `range(s, s + dias)` |
| $\sum_{(v,t)} t\, x_{pvt}$ | "suma de t por x" | el día en que empieza $p$: solo un $x$ vale 1 | `sum(t * x[p, v, t] ...)` |
| $\ell_p$ | "ele de p" | duración en días | `dias` |

**Fíjate:** la restricción de barco es **una fila por barco y día**: "ese día el barco está en a lo más un proyecto".
No hay pares, no hay M. Es la misma idea de la cobertura de tu red de turnos: un bloque, una fila.

In [ ]:
def barcos_milp(K, proyectos, precedencias=(), relajar=False, limite=60):
    claves = list(zip(K["proyecto"], K["barco"], K["t"]))
    costo = dict(zip(claves, K["costo"]))
    dur = dict(zip(claves, K["dias"]))
    aplazar = proyectos.set_index("proyecto")["costo_aplazar"].to_dict()
    P = list(proyectos["proyecto"])

    por_proyecto, ocupa = defaultdict(list), defaultdict(list)
    for k in claves:
        por_proyecto[k[0]].append(k)
        for dia in range(k[2], k[2] + dur[k]):
            ocupa[k[1], dia].append(k)  # (barco, día) → inicios que lo ocupan ese día

    tipo = pyo.UnitInterval if relajar else pyo.Binary
    m = pyo.ConcreteModel()
    m.K = pyo.Set(initialize=claves, dimen=3)
    m.x = pyo.Var(m.K, within=tipo)
    m.u = pyo.Var(P, within=tipo)
    m.obj = pyo.Objective(expr=sum(costo[k] * m.x[k] for k in m.K) + sum(aplazar[p] * m.u[p] for p in P),
                          sense=pyo.minimize)
    m.una_vez = pyo.Constraint(P, rule=lambda m, p: sum(m.x[k] for k in por_proyecto[p]) + m.u[p] == 1)
    m.barco_dia = pyo.Constraint(list(ocupa), rule=lambda m, v, dia: sum(m.x[k] for k in ocupa[v, dia]) <= 1)
    m.prec = pyo.Constraint(list(precedencias), rule=lambda m, a, b:
                            sum(k[2] * m.x[k] for k in por_proyecto[b]) >= sum((k[2] + dur[k]) * m.x[k] for k in por_proyecto[a]))

    inicio = time.perf_counter()
    res = SOLVER.solve(m, timelimit=limite)
    m.segundos = time.perf_counter() - inicio
    m.cota = res.problem.lower_bound
    return m


mb = barcos_milp(K, proyectos, PRECEDENCIAS)
lpb = barcos_milp(K, proyectos, PRECEDENCIAS, relajar=True)
print("MILP:", eur(pyo.value(mb.obj)), f"en {mb.segundos:.2f} s")
print("LP:  ", eur(pyo.value(lpb.obj)))
print("Tamaño:", tamano(mb))

In [ ]:
elegidos = [k for k in mb.K if pyo.value(mb.x[k]) > 0.5]
plan_b = K.set_index(["proyecto", "barco", "t"]).loc[elegidos].reset_index()
plan_b = plan_b.assign(inicio=plan_b["t"], fin=plan_b["t"] + plan_b["dias"], trabajo=plan_b["proyecto"])
print("Aplazados:", [p for p in proyectos["proyecto"] if pyo.value(mb.u[p]) > 0.5] or "ninguno")
plan_b[["proyecto", "barco", "inicio", "fin", "entrega", "retraso", "costo"]].sort_values(["barco", "inicio"])

In [ ]:
gantt(plan_b, "barco", "Barcos: plan de 36 días (gris = mal tiempo)", cerrado=MAL_TIEMPO, ancho=12, eje="días")

**Lectura:**
- El plan cuesta **3.210.000 €**. La cota del LP es la **misma**: la relajación ya da la solución entera. Compara con el job shop: cota 7 contra óptimo 14.
- El jack-up hace las dos cimentaciones (es más barato que el barco grúa) y las turbinas de BW1.
- Las **turbinas de BW1 llegan 11 días tarde**: 660.000 € de multa. Solo el jack-up las puede montar, y necesitan 8 días seguidos de buen tiempo.
  Entre los días 18 y 24 hay 7 días buenos, no 8. El primer hueco de 8 días empieza el 27.
- La cimentación de BW1 ocupa los días 0 a 5, antes de que las turbinas estén listas (día 8): la precedencia se cumple sin costo.
- La grúa hace la reparación urgente de BW3 y la subestación. El barco de cable hace los tres cables, todos a tiempo.

**Por qué esta formulación es fuerte.** Cada fila "barco $v$, día $t$" mira todos los proyectos que podrían estar ahí ese día.
Con $x = 0{,}5$ no hay forma de "medio esconder" un choque: dos mitades el mismo día suman 1, el máximo.
En la disyuntiva con big-M, dos mitades se aflojaban $M/2$ cada una.

**El precio:** el modelo crece con el **horizonte**. Una binaria por día posible. Con días está bien; con minutos y un año, explota.

### 7.2 El modelo crece: hasta 80 proyectos y 10 barcos
Generamos carteras al azar: duraciones de 2 a 8 días, cada barco sirve para la mitad de los proyectos, 1 de cada 6 días con mal tiempo.
Si no caben todos, el modelo aplaza (a 3.000.000 € cada uno).

In [ ]:
def cartera_aleatoria(n_proy, n_barcos, dias, semilla=1):
    rng = np.random.default_rng(semilla)
    proy = pd.DataFrame({"proyecto": [f"P{i:02d}" for i in range(n_proy)], "dias": rng.integers(2, 9, n_proy),
                         "listo": rng.integers(0, dias // 2, n_proy), "multa_dia": rng.integers(20, 100, n_proy) * 1000,
                         "costo_aplazar": 3_000_000})
    proy["entrega"] = proy["listo"] + proy["dias"] + rng.integers(0, 15, n_proy)
    barc = pd.DataFrame({"barco": [f"V{i}" for i in range(n_barcos)], "costo_dia": rng.integers(30, 100, n_barcos) * 1000})
    comp = pd.DataFrame([(p, v) for p in proy["proyecto"] for v in barc["barco"] if rng.random() < 0.5],
                        columns=["proyecto", "barco"])
    malo = set(rng.choice(dias, dias // 6, replace=False).tolist())
    clim = pd.DataFrame({"dia": range(dias), "ventana": [t not in malo for t in range(dias)]})
    return proy, barc, comp, clim


filas = []
for n_proy, n_barcos, dias in [(10, 3, 40), (20, 4, 60), (40, 6, 90), (80, 10, 120)]:
    proy, barc, comp, clim = cartera_aleatoria(n_proy, n_barcos, dias)
    Kn = inicios_factibles(proy, barc, comp, clim)
    mn = barcos_milp(Kn, proy)
    ln = barcos_milp(Kn, proy, relajar=True)
    filas.append({"proyectos": n_proy, "barcos": n_barcos, "dias": dias,
                  "todos_pvt": n_proy * n_barcos * dias, "binarias": len(Kn) + n_proy,
                  "milp": eur(pyo.value(mn.obj)), "lp": eur(pyo.value(ln.obj)),
                  "gap_lp_%": round(100 * (pyo.value(mn.obj) - pyo.value(ln.obj)) / pyo.value(mn.obj), 2),
                  "segundos": round(mn.segundos, 2), "aplazados": round(sum(pyo.value(mn.u[p]) for p in proy["proyecto"]))})
pd.DataFrame(filas)

**Cómo se lee:**
- La distancia entre el LP y el MILP (`gap_lp_%`) queda **por debajo de 0,1 %** en todos los tamaños. En el job shop con big-M, la cota quedaba a 87 %.
- Con 80 proyectos, 10 barcos y 120 días hay más de 11.000 binarias, y resuelve en menos de un segundo.
- `todos_pvt` contra `binarias`: generar solo lo factible deja entre el 9 % y el 23 % de las variables.
- En todas las carteras se aplaza algo: hay proyectos que no caben, o que ningún barco puede hacer sin pisar mal tiempo.
  El modelo elástico dice **cuáles y cuántos**. Es la demanda no atendida de tu red de Siigo, en barcos.

**Los dos tamaños.** Modelo: hasta $|P| \cdot |V| \cdot H$ binarias, lineal en cada uno. Espacio de búsqueda: cada proyecto elige uno de sus $(v, t)$:
del orden de $(|V| \cdot H)^{|P|}$. Con 80 proyectos, $(10 \cdot 120)^{80} \approx 10^{246}$.

## 8. ¿MILP o CP?
Una sola regla para empezar:

> **Si el problema es sobre todo orden en recursos compartidos, empieza con CP-SAT. Si el tiempo es grueso (días, turnos) y los costos dependen del momento, usa un MILP indexado en el tiempo.**

| Señal en el problema | Herramienta | Por qué |
|---|---|---|
| Orden en máquinas, makespan, retrasos, muchas disyuntivas | CP-SAT | `NoOverlap` razona con la carga; el big-M es débil |
| Tiempo en días o turnos, horizonte corto | MILP indexado en el tiempo | relajación fuerte; clima y multas se precalculan |
| Tiempo en minutos y horizonte largo | CP-SAT | el indexado explota: una variable por minuto |
| Cantidades continuas, flujos, inventarios, precios sombra | MILP | CP-SAT solo trabaja con enteros y no da duales |
| Reglas raras ("si A usa la grúa el lunes, B no puede el martes") | CP-SAT | acepta lógica directa, sin linealizar |
| La empresa ya tiene Gurobi o CPLEX y un modelo grande de costos | MILP | integrarse con lo que hay pesa más |

**En la práctica:** se prueban los dos con una instancia chica y se mide duplicando (lección 6). Para scheduling de taller, CP-SAT casi siempre gana.

## 9. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Secuenciación | Decidir el orden de los trabajos en un recurso | banco de presión, taller, barcos | scheduling, sequencing |
| Retraso | $\max\{0, C_j - d_j\}$: cuánto pasa de la entrega | 6 h en el banco | tardiness |
| Makespan | Hora en que termina el último trabajo | 14 h en el taller | makespan |
| Disyuntiva | "i antes que j, o j antes que i" | una binaria por par | disjunctive constraint |
| Big-M débil | La relajación afloja las dos ramas a la vez | cota LP 0 h contra óptimo 6 h | weak LP relaxation |
| Job shop | Cada trabajo con su ruta por las máquinas | 5 trabajos × 3 máquinas | job shop scheduling |
| Cota simple | La máquina más cargada o la ruta más larga | 13 h, mejor que el LP (7 h) | combinatorial lower bound |
| Programación por restricciones | Tachar valores imposibles y aprender de los fracasos | CP-SAT | constraint programming |
| Variable de intervalo | Inicio, duración y fin en un solo objeto | `new_interval_var` | interval variable |
| NoOverlap | Intervalos que no se pisan | una línea por máquina | no-overlap (disjunctive) constraint |
| Cuello de botella | Recurso al 100 %: fija el makespan | la máquina más cargada con 40 trabajos | bottleneck |
| Indexado en el tiempo | Una binaria por (tarea, recurso, momento) | $x_{pvt}$ de los barcos | time-indexed formulation |
| Costo precalculado | Todo lo que depende del día entra como dato | clima, multas | precomputed cost |

### Notación de hoy
| Símbolo | Significa | En la lección |
|---|---|---|
| $S_j$, $C_j = S_j + p_j$ | inicio y fin de un trabajo | `S[j]` |
| $T_j \ge C_j - d_j$, $T_j \ge 0$ | retraso, linealizado | 6 h |
| $\lor$ | "o": se cumple al menos una | la disyuntiva |
| $n!$ | factorial: número de órdenes | $4! = 24$, $20! \approx 2{,}4 \cdot 10^{18}$ |
| $\binom{n}{2} = n(n-1)/2$ | número de pares | 6 binarias con 4 trabajos |
| $\prod_{k \in K} n_k!$ | producto: multiplicar sobre un conjunto | órdenes del job shop |
| $C_{\max}$ | makespan | 14 h |
| $D = \{(a, b) : m_a = m_b\}$ | pares que comparten máquina | 22 pares |
| $[a, b)$ | intervalo cerrado en $a$, abierto en $b$ | `new_interval_var` |
| $\text{NoOverlap}(\cdot)$ | restricción global de CP | `add_no_overlap` |
| $\{0, 1, \dots, H\}$ | dominio entero | `new_int_var(0, H)` |
| $x_{pvt}$ | binaria indexada en el tiempo | proyecto, barco, día |
| $s \le t < s + \ell_p$ | el proyecto sigue en el barco el día $t$ | `range(s, s + dias)` |

## 10. Ejercicios
1. **Mantenimiento del torno.** El torno para de la hora 4 a la 6. Antes de resolver, adivina: ¿el makespan de 14 h sube? ¿cuánto?
   Usa `jobshop_cp(OPS, paradas={"torno": [(4, 6)]})` y mira el Gantt.
2. **Un M demasiado chico.** Resuelve `una_maquina(TRABAJOS, M=13)`: una hora menos que el horizonte. Antes, adivina: ¿da 6 h, menos, o falla? ¿Por qué?
3. **Tormenta en el Báltico.** Los días 18 a 22 también son de mal tiempo. Re-resuelve los barcos. ¿Cuánto sube el costo? ¿Qué proyectos se mueven?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"How would you model a job shop or a vessel scheduling problem? MILP or constraint programming?"*
> *"It depends on what dominates. If it's mostly sequencing on shared resources, with makespan or tardiness, I'd start with CP-SAT:
> interval variables and no-overlap constraints propagate machine load directly. The MILP version needs a big-M disjunction per pair,
> and its LP relaxation is weak: with fractional binaries both orders are relaxed at once, so the bound barely moves.
> If time is coarse, like days for offshore vessels, and costs depend on the day, like weather windows or penalties,
> I'd use a time-indexed MILP. It has many more binaries, but the relaxation is very tight, and I generate only feasible start times."*

Seguimiento: *"Why is the big-M formulation weak?"*
> *"With y = 0.5, each of the two disjunctive constraints is relaxed by M/2. If M is the horizon, that's larger than any processing time,
> so the LP lets all jobs overlap. In my single-machine example the LP bound was 0 against an optimum of 6, whatever M I used.
> A simple bound like the load of the busiest machine was much better than the LP bound."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
parada = jobshop_cp(OPS, paradas={"torno": [(4, 6)]})
print(f"Con parada: {parada['estado']}, makespan {parada['makespan']:.0f} h (antes {cp['makespan']:.0f} h)")
gantt(parada["plan"], "maquina", f"Torno parado de 4 a 6: makespan {parada['makespan']:.0f} h")

El makespan sube de 14 a **17 h**: 3 h, más que las 2 h de la parada.
La parada cae en medio de la mañana y obliga a reordenar: las rutas que pasaban por el torno a esa hora esperan, y la espera se propaga a las otras máquinas.
Con CP-SAT fue una línea: un intervalo fijo dentro del `NoOverlap` del torno. En el MILP habría que agregar la parada como una operación fija y sus disyuntivas con big-M.

In [ ]:
# Ejercicio 2
for M in [13, 14]:
    try:
        print(f"M = {M}: retraso {pyo.value(una_maquina(TRABAJOS, M=M).obj):.0f} h")
    except RuntimeError:  # HiGHS no encontró ninguna solución que cargar
        print(f"M = {M}: el modelo es infactible")

Con M = 13 el modelo es **infactible**, aunque los 24 órdenes del banco son posibles.
Cualquier orden tiene un primer trabajo (empieza en 0) y un último (termina en 14). Para ese par, la restricción que debería "apagarse" dice
$S_{\text{primero}} \ge C_{\text{último}} - M = 14 - 13 = 1$: el primero no podría empezar en 0. M tiene que cubrir la distancia más larga posible: el horizonte.
Aquí el error se ve, porque el modelo falla. En modelos grandes, un M chico puede cortar solo algunos planes, el óptimo entre ellos, y entregar uno peor **sin avisar** (lección 3).

In [ ]:
# Ejercicio 3
clima_tormenta = clima.assign(ventana=clima["ventana"] & ~clima["dia"].between(18, 22))
K3 = inicios_factibles(proyectos, barcos, compat, clima_tormenta)
m3 = barcos_milp(K3, proyectos, PRECEDENCIAS)
print("Costo con tormenta:", eur(pyo.value(m3.obj)), "| antes:", eur(pyo.value(mb.obj)),
      "| sube:", eur(pyo.value(m3.obj) - pyo.value(mb.obj)))
nuevo = {k[0]: (k[1], k[2]) for k in m3.K if pyo.value(m3.x[k]) > 0.5}
antes = {k[0]: (k[1], k[2]) for k in mb.K if pyo.value(mb.x[k]) > 0.5}
pd.DataFrame({"antes (barco, día)": antes, "tormenta (barco, día)": nuevo}).loc[lambda t: t.iloc[:, 0] != t.iloc[:, 1]]

La tormenta cuesta **150.000 €**. Solo se mueve el cable de exportación de BW2: del día 18 al 27.
Termina el día 33 en vez del 30: 3 días de multa a 50.000 €. Los demás proyectos ya estaban fuera de los días 18 a 22.
Es sensibilidad por re-solve (lección 3): cambias el dato, regeneras los inicios factibles y resuelves. Con el modelo indexado, el clima nuevo es solo un filtro distinto.